# 11 — MCP Tools

**Mode:** Services  
**Level:** Advanced  
**Estimated time:** 40 minutes

This notebook is meant to be read and run from top to bottom. Each code cell
changes one small part of the system, and the inspection cells show the real
Praval objects produced by that change.


## What you will build

Two real official-SDK MCP servers: a spawned stdio server and a local Streamable HTTP server. You will discover namespaced schemas, attach async-only tools to an Agent, approve a real MCP invocation through HITL, normalize a structured result, bound a timeout, and verify cleanup.


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "examples" / "notebooks",
]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, get_events, require_env, setup_notebook,
    show_callout, show_flow, show_json, show_roles, show_spore,
    show_timeline, stage,
)

setup_notebook(11, 'MCP Tools')


## Prerequisites and setup

**Course prerequisites:** Complete `course-05-tool-use`, `course-09-model-runtime`, `course-10-hitl`.

**Execution requirements:** Services. Install `praval[mcp]`. No paid provider key is used. The lesson starts official-SDK servers from `examples/certification/mcp_server.py` and permits loopback HTTP for the local Streamable HTTP contract.

Run the setup cell above first. It configures presentation helpers and a
credential-free lifecycle provider. It does not hide any agent workflow.


## Learning goals

- Connect to stdio and Streamable HTTP MCP servers.
- Convert discovered schemas to namespaced ToolSpec objects.
- Run an async-only MCP handler through Agent HITL approval.
- Normalize text/structured results, timeout, and close all processes.


## Mental model

MCP is a protocol boundary for external tools. `MCPClient` negotiates a server session, calls `tools/list`, prefixes names to prevent collisions, and attaches async handlers through `Agent.add_tool_spec()`. The Agent still owns approval and ModelRuntime execution. Provider-hosted MCP descriptors are a separate provider capability and do not create a Praval MCPClient session.


In [ ]:
show_flow(
('MCP server', 'tools/list', 'provider'),
('MCPClient', 'namespace and normalize', 'reef'),
('Agent', 'async tool and approval', 'agent'),
('tools/call', 'ToolResult', 'tool'),
)


## Try it

We will now assemble the workflow in small steps. Run each cell, then pause at the inspection that follows it.


### Prepare the official SDK server asset

The committed server implements deterministic echo, status, and slow tools. Small socket helpers choose and wait for a local HTTP port.


In [ ]:
import asyncio
import socket
import subprocess
import sys
import tempfile
import time

from praval import Agent, InterventionRequired, get_provider_registry
from praval.mcp import MCPClient, MCPServerConfig
from praval.models import (
    ModelResponse, ProviderCapabilities, ToolCall,
)

server_path = find_example_asset("certification/mcp_server.py")


def free_port():
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


def wait_for_port(port, timeout=10):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        with socket.socket() as sock:
            sock.settimeout(0.1)
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return
        time.sleep(0.05)
    raise RuntimeError("MCP Streamable HTTP server did not start")


### What just happened?

The server path resolves in both a source checkout and exact-wheel isolated certification workspace.

### Why this matters

Contract examples should use protocol servers, not direct calls to the tool functions.


### Define a deterministic tool-calling provider

This local provider supplies the model decision in a service-mode lesson. Its tool call still travels through ModelRuntime, HITL, and the real MCP session.


In [ ]:
class MCPApprovalProvider:
    provider_name = "course-mcp-runtime"
    capabilities = ProviderCapabilities(text=True, tools=True)

    def __init__(self, config):
        self.config = config

    async def ainvoke(self, request, tools=None):
        return ModelResponse(tool_calls=[ToolCall(
            id="course-mcp-call", name="notebook_stdio__echo",
            arguments={"message": "approved-praval"},
        )])

    def continue_with_tool_results(self, request, response, results):
        return ModelResponse(
            content=f"MCP returned {results[0].content}",
            provider=self.provider_name, model=request.model,
        )


get_provider_registry().register_provider(
    "course-mcp-runtime", MCPApprovalProvider,
    default_model="course-mcp-model",
    capabilities=MCPApprovalProvider.capabilities,
)


### What just happened?

The adapter emits one provider-neutral ToolCall and knows how to continue after a ToolResult. It does not implement the MCP tool itself.

### Why this matters

Infrastructure fakes may drive orchestration, but they cannot replace the real MCP server, transport, discovery, invocation, or cleanup being certified.


### Discover and approve stdio tools

The context manager starts the child process without a shell, performs discovery, and stays in one asynchronous task through approval, invocation, and close. Discovery prefixes every tool with the server name.


In [ ]:
stdio_config = MCPServerConfig(
    name="notebook_stdio", transport="stdio",
    command=sys.executable, args=[str(server_path)],
    require_approval=True, tool_timeout=30,
)
hitl_workspace = tempfile.TemporaryDirectory(prefix="praval-mcp-hitl-")
hitl_db = str(Path(hitl_workspace.name) / "mcp-hitl.sqlite3")
mcp_agent = Agent(
    "course-mcp-agent", provider="course-mcp-runtime",
    model="course-mcp-model", hitl_enabled=True, hitl_db_path=hitl_db,
)

async with MCPClient(stdio_config) as stdio_client:
    specs = await stdio_client.register_tools(mcp_agent)
    names = {spec.name for spec in specs}
    assert "notebook_stdio__echo" in names
    assert all(spec.requires_approval for spec in specs)
    assert mcp_agent.tools["notebook_stdio__echo"]["async_only"] is True
    stage("MCP stdio", "tools discovered", f"{len(specs)} tools")
    try:
        await mcp_agent.agenerate("Call the approved MCP echo tool.")
    except InterventionRequired as exc:
        approval_run_id = exc.run_id
        approval_intervention_id = exc.intervention_id
        pending = mcp_agent.get_pending_interventions(run_id=approval_run_id)
        assert len(pending) == 1
        stage("HITL", "MCP call interrupted", pending[0].tool_name)
    else:
        raise AssertionError("Expected MCP approval interruption")
    mcp_agent.approve_intervention(
        approval_intervention_id, reviewer="notebook-operator"
    )
    resumed = await mcp_agent.aresume_run(approval_run_id)
    assert "echo:approved-praval" in resumed
    stage("MCP stdio", "approved tool called", resumed)
assert not stdio_client.connected
stage("MCP stdio", "closed", "spawned process exited")


### What just happened?

The Agent contains ordinary ToolSpec-backed entries. Namespacing prevents an `echo` tool from this server colliding with `echo` from another server.

### Why this matters

MCP adapts into Praval's existing tool registry rather than bypassing tool policy.


In [ ]:
show_json(
    [spec.model_dump(exclude_none=True) for spec in specs],
    "Namespaced MCP ToolSpecs",
)


### Inspect the approved Agent invocation

The provider-generated call interrupted before execution, the persisted intervention was approved, and resume invoked the real MCP tool. Inspect the evidence retained from that lifecycle.


In [ ]:
show_json(
    {
        "run_id": approval_run_id,
        "tool": pending[0].tool_name,
        "response": resumed,
        "client_closed": not stdio_client.connected,
    },
    "Approved MCP execution",
)


### What just happened?

Approval resumed on the notebook event loop, invoked the real stdio server, and returned its text through a normalized ToolResult.

### Why this matters

Async-only MCP sessions must remain on their owning loop; sync Agent APIs reject them with guidance to use `agenerate()` or `astream()`.


### Use Streamable HTTP and structured content

A local official-SDK server exposes the same protocol over the recommended remote transport. The structured status payload remains in ToolResult metadata.


In [ ]:
port = free_port()
process = subprocess.Popen(
    [sys.executable, str(server_path), "http", str(port)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
try:
    wait_for_port(port)
    http_config = MCPServerConfig(
        name="notebook_http", transport="streamable_http",
        url=f"http://127.0.0.1:{port}/mcp", require_approval=False,
    )
    async with MCPClient(http_config) as http_client:
        http_specs = await http_client.list_tools()
        structured = await http_client.call_tool(
            "notebook_http__status", {"component": "notebook"}
        )
        assert structured.metadata["structured_content"] == {
            "component": "notebook", "status": "ready",
        }
        assert any(spec.name == "notebook_http__status" for spec in http_specs)
    assert not http_client.connected
finally:
    process.terminate()
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)
stage("MCP HTTP", "structured result", structured.content)


### What just happened?

The Streamable HTTP session negotiated, discovered tools, returned structured content, closed its network session, and stopped the server process.

### Why this matters

Normalized metadata preserves machine-readable content without forcing it into display text.


In [ ]:
show_json(structured.model_dump(exclude_none=True), "Structured MCP ToolResult")


### Bound slow execution

A second stdio client uses a one-second tool timeout against a two-second server call. Timeout becomes a structured tool error and its context still closes safely.


In [ ]:
timeout_config = MCPServerConfig(
    name="notebook_timeout", transport="stdio",
    command=sys.executable, args=[str(server_path)],
    require_approval=False, tool_timeout=1.0,
)
async with MCPClient(timeout_config) as timeout_client:
    timeout_result = await timeout_client.call_tool(
        "notebook_timeout__slow", {"seconds": 2.0}
    )
    assert timeout_result.is_error
    assert "timed out" in timeout_result.content
    stage("MCP", "timeout bounded", timeout_result.content)
show_timeline()


### What just happened?

The slow server did not hold the workflow indefinitely, and the spawned stdio process exited when its context closed.

### Why this matters

Timeout and cleanup rules are part of a remote tool's correctness contract.


## Your turn

Connect a second stdio client with a different server name. Confirm both expose an `echo` capability without a name collision, then close both contexts.


In [ ]:
# second = MCPServerConfig(
#     name="second_stdio", transport="stdio",
#     command=sys.executable, args=[str(server_path)],
# )
# Expect notebook_stdio__echo and second_stdio__echo as distinct names.


## Common mistake

**Mistake:** Calling an async-only MCP tool through `Agent.generate()`.

**Correction:** Use `await Agent.agenerate()` or `await Agent.astream()` so the MCP session is not moved across event loops.


<details>
<summary>Under the hood</summary>

The MCP SDK owns protocol framing and session negotiation. MCPClient validates transport security, discovers schemas, normalizes ToolSpecs, enforces size and time bounds, converts supported result blocks, and closes the underlying stream or process. Praval's ModelRuntime and HITL remain the execution policy layer.

</details>


## Recap

- Praval 0.8 supports MCP tools over stdio and Streamable HTTP.
- Discovered tools are namespaced and registered as async-only ToolSpecs.
- Approval is enforced by the normal Agent HITL path.
- Timeouts, structured results, and cleanup are visible outcomes.


### Next lesson

Continue to `12_voice_and_multimodal.ipynb` for a real speech round trip and multimodal model input, and to distinguish request-based media from realtime sessions.


## Cleanup

Release every agent, transport, temporary file, or external resource created by this lesson. Cleanup is part of the example, not an afterthought.


In [ ]:
mcp_agent.close()
hitl_workspace.cleanup()
stage("MCP", "cleanup complete", "client, agent, and HITL store released")
